# Stitch layout model – train in Colab

This notebook trains a small **implicit** model: given tile images from your stitch exports, it predicts the layout (normalized position, size, rotation) for each tile. Uses your exported training ZIPs (`controls.json` + `tiles/*.png`).

**Steps:**
1. Setup (GPU, PyTorch)
2. Get your training data into Colab
3. Dataset: load ZIPs and parse controls
4. Model: CNN + set aggregation + per-tile heads
5. Training loop
6. Quick check: predict on one sample

---
## Step 1: Setup

Run this first. Enable GPU: **Runtime → Change runtime type → T4 GPU** (or any GPU), then run the cell.

In [ ]:
!pip install torch torchvision --quiet

import json
import zipfile
import os
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Step 2: Get your training data into Colab

**Option A – Upload a ZIP of training bundles:**  
On your computer, put all your `stitch-training-*.zip` files into one folder, then zip that folder into `training_bundles.zip`. In Colab, run the cell below and use the file picker to upload `training_bundles.zip`. We'll unzip it to `/content/training_bundles`.

**Option B – Google Drive:**  
Put your training ZIPs in a folder on Drive (e.g. `MyDrive/stitch_training`). Uncomment the Drive mount and set `DATA_ROOT` to that folder path.

In [ ]:
# Option A: Upload a zip file. Either:
# - A single stitch-training-YYYYMMDD-HHmmss.zip, or
# - A zip containing multiple stitch-training-*.zip files, or
# - A zip containing already-extracted bundle folders (each with controls.json + tiles/)
from google.colab import files

uploaded = files.upload()  # Pick your zip
zip_path = list(uploaded.keys())[0]

DATA_ROOT = "/content/training_bundles"
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_ROOT)

# If the zip contained other .zip files (e.g. stitch-training-*.zip), extract those too
for root, _, filenames in os.walk(DATA_ROOT):
    for name in filenames:
        if name.endswith(".zip"):
            path = os.path.join(root, name)
            out_dir = os.path.join(root, name[:-4])
            with zipfile.ZipFile(path, "r") as z:
                z.extractall(out_dir)

def find_bundle_dirs(root: str) -> List[str]:
    out = []
    for dirpath, _, filenames in os.walk(root):
        if "controls.json" in filenames:
            out.append(dirpath)
    return out

bundle_dirs = find_bundle_dirs(DATA_ROOT)
print(f"Found {len(bundle_dirs)} training bundles.")
if bundle_dirs:
    print("First bundle:", bundle_dirs[0])

In [ ]:
# If Option A left you with one folder containing many stitch-training-* subfolders:
# bundle_dirs = [os.path.join(DATA_ROOT, d) for d in os.listdir(DATA_ROOT)
#               if os.path.isdir(os.path.join(DATA_ROOT, d)) and os.path.isfile(os.path.join(DATA_ROOT, d, "controls.json"))]
# If you uploaded a SINGLE stitch-training-*.zip, DATA_ROOT might already be the bundle; then:
if not bundle_dirs and os.path.isfile(os.path.join(DATA_ROOT, "controls.json")):
    bundle_dirs = [DATA_ROOT]
    print("Using single bundle at", DATA_ROOT)
print(f"Total bundles: {len(bundle_dirs)}")

---
## Step 3: Dataset

Load each bundle: read `controls.json`, load `tiles/tile_0.png`, `tile_1.png`, ... in order. Return images as a tensor and targets (normalized x, y, w, h, rotation) for each tile. We support up to `max_tiles` (pad with zeros and a valid mask).

In [ ]:
IMG_SIZE = 224  # Resize each tile to this for the CNN
MAX_TILES = 8   # Max number of tiles per sample; pad if fewer

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet
])

class StitchDataset(Dataset):
    def __init__(self, bundle_dirs: List[str], max_tiles: int = MAX_TILES, transform=None):
        self.bundle_dirs = bundle_dirs
        self.max_tiles = max_tiles
        self.transform = transform or (lambda x: x)

    def __len__(self) -> int:
        return len(self.bundle_dirs)

    def __getitem__(self, idx: int):
        root = self.bundle_dirs[idx]
        with open(os.path.join(root, "controls.json"), "r") as f:
            controls = json.load(f)

        tiles_meta = controls["tiles"]
        n = len(tiles_meta)
        assert n <= self.max_tiles, f"Too many tiles ({n}) in {root}"

        # Load tile images
        images = []
        for i in range(self.max_tiles):
            path = os.path.join(root, "tiles", f"tile_{i}.png")
            if i < n and os.path.isfile(path):
                img = Image.open(path).convert("RGB")
                images.append(self.transform(img))
            else:
                # Pad with black image
                images.append(torch.zeros(3, IMG_SIZE, IMG_SIZE))

        # Stack: (max_tiles, 3, H, W)
        images = torch.stack(images)

        # Targets: (max_tiles, 5) -> nx, ny, nw, nh, rotation (degrees / 360 for scale)
        targets = torch.zeros(self.max_tiles, 5)
        valid = torch.zeros(self.max_tiles, dtype=torch.float32)
        for i in range(n):
            t = tiles_meta[i]
            norm = t.get("normalized", {})
            targets[i, 0] = norm.get("x", 0)
            targets[i, 1] = norm.get("y", 0)
            targets[i, 2] = norm.get("width", 0)
            targets[i, 3] = norm.get("height", 0)
            rot = t.get("rotation", 0)
            targets[i, 4] = (rot % 360) / 360.0  # 0-1
            valid[i] = 1.0

        return {
            "images": images,
            "targets": targets,
            "valid": valid,
            "n_tiles": n,
        }

dataset = StitchDataset(bundle_dirs, transform=transform)
print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print(f"Images shape: {sample['images'].shape}, targets shape: {sample['targets'].shape}, valid: {sample['valid']}")

---
## Step 4: Model

Per-tile CNN backbone → set aggregation (transformer) → per-tile MLP heads → 5 values per tile (nx, ny, nw, nh, rotation).

In [ ]:
class LayoutModel(nn.Module):
    def __init__(self, max_tiles: int = MAX_TILES, backbone_name: str = "resnet18", dim: int = 256, num_heads: int = 4, num_layers: int = 2):
        super().__init__()
        self.max_tiles = max_tiles
        # Backbone: each tile (B, 3, H, W) -> (B, feat_dim). We'll run per tile.
        from torchvision.models import resnet18, ResNet18_Weights
        resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC
        backbone_out = 512  # ResNet18 last conv channels
        self.proj = nn.Linear(backbone_out, dim)

        # Set encoder: transformer over tile embeddings
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=num_heads, dim_feedforward=dim * 2, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Per-tile head: dim -> 5 (nx, ny, nw, nh, rot_01)
        self.head = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, 5),
            nn.Sigmoid(),  # All outputs 0-1
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # images: (B, max_tiles, 3, H, W)
        B, N, C, H, W = images.shape
        x = images.view(B * N, C, H, W)
        x = self.backbone(x)  # (B*N, 512, 1, 1)
        x = x.flatten(1)      # (B*N, 512)
        x = self.proj(x)     # (B*N, dim)
        x = x.view(B, N, -1)

        x = self.transformer(x)  # (B, N, dim)
        out = self.head(x)      # (B, N, 5)
        return out

model = LayoutModel(max_tiles=MAX_TILES).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total / 1e6:.2f}M")

---
## Step 5: Training loop

Loss: L1 on predicted (nx, ny, nw, nh, rot) vs targets, masked by `valid`. Train for a few epochs.

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        images = batch["images"].to(device)
        targets = batch["targets"].to(device)
        valid = batch["valid"].to(device)  # (B, max_tiles)

        pred = model(images)  # (B, N, 5)
        loss_per = torch.abs(pred - targets).sum(dim=-1)  # (B, N)
        loss = (loss_per * valid).sum() / (valid.sum() * 5 + 1e-8)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-4

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    loss = train_epoch(model, loader, optimizer, device)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss: {loss:.4f}")

---
## Step 6: Check predictions

Run the model on one sample and compare predicted vs target layout.

In [ ]:
model.eval()
with torch.no_grad():
    sample = dataset[0]
    images = sample["images"].unsqueeze(0).to(device)
    pred = model(images).squeeze(0).cpu()  # (max_tiles, 5)
    targets = sample["targets"]
    valid = sample["valid"]

n = int(sample["n_tiles"])
print("Tile  Predicted (nx, ny, nw, nh, rot)     Target")
for i in range(n):
    p = pred[i].numpy()
    t = targets[i].numpy()
    rot_deg = p[4] * 360
    t_rot_deg = t[4] * 360
    print(f" {i}   ({p[0]:.3f}, {p[1]:.3f}, {p[2]:.3f}, {p[3]:.3f}, {rot_deg:.1f}°)   ({t[0]:.3f}, {t[1]:.3f}, {t[2]:.3f}, {t[3]:.3f}, {t_rot_deg:.1f}°)")

In [ ]:
# Save the model so you can download it and use it later
torch.save({
    "model_state": model.state_dict(),
    "max_tiles": MAX_TILES,
    "img_size": IMG_SIZE,
}, "/content/stitch_layout_model.pt")
print("Saved to /content/stitch_layout_model.pt")
print("Download: Files panel on the left, or run files.download('/content/stitch_layout_model.pt')")
files.download("/content/stitch_layout_model.pt")